In [1]:
import pandas as pd

df = pd.read_excel("Sultanlar_Ligi_2023-24_matches.xlsx", sheet_name="Matches")
df.head()

,match_id,season,phase,round,date,time,venue,home,away,home_sets,...,S4_A,S5_H,S5_A,home_pts,away_pts,sets_played,check,home_id,away_id,shared_hall
0,2324-001,2023-24,League,1,2023-10-14,13:00,TVF Ziraat Bankkart Voleybol Salonu,PTT,KUZEYBORU,1,...,27.0,NaN,NaN,82,96,4,OK,PTT,KUZ,0
1,2324-002,2023-24,League,1,2023-10-14,14:00,Aski Atatürk,ÇUKUROVA BLD.,ANKARAGÜCÜ,0,...,NaN,NaN,NaN,67,75,3,OK,CUK,ANK,0
2,2324-003,2023-24,League,1,2023-10-14,14:00,Vakıfbank Spor Sarayı,VAKIFBANK,KARAYOLLARI,3,...,NaN,NaN,NaN,76,61,3,OK,VB,KAR,0
3,2324-004,2023-24,League,1,2023-10-14,15:30,Burhan Felek Vestel Voleybol Salonu,FENERBAHÇE OPET,AYDIN B.ŞEHİR BLD.,3,...,NaN,NaN,NaN,79,61,3,OK,FB,AYD,0
4,2324-005,2023-24,League,1,2023-10-14,17:00,Bahçeköy Orman Fakültesi,SARIYER BLD.,BEŞİKTAŞ AYOS,2,...,21.0,13.0,15.0,99,99,5,OK,SAR,BJK,0


In [2]:
S_MAP = {(3,0):1.0, (3,1):0.8, (3,2):0.6, (2,3):0.4, (1,3):0.2, (0,3):0.0}

def run_elo(df, K=30, H=50, start=1500):
    ratings = {}
    pre_home, pre_away, probs = [], [], []
    for _, m in df.sort_values("date").iterrows():
        rh = ratings.get(m.home_id, start)
        ra = ratings.get(m.away_id, start)
        h = 0 if m.shared_hall else H
        e = 1 / (1 + 10 ** ((ra - (rh + h)) / 400))
        pre_home.append(rh); pre_away.append(ra); probs.append(e)   # store BEFORE update
        s = S_MAP[(m.home_sets, m.away_sets)]
        d = K * (s - e)
        ratings[m.home_id] = rh + d
        ratings[m.away_id] = ra - d
    out = df.sort_values("date").copy()
    out["elo_home"], out["elo_away"], out["p_home"] = pre_home, pre_away, probs
    return out, ratings


In [3]:
matches_elo, ratings = run_elo(df)

team_names = (
    pd.read_excel("Sultanlar_Ligi_2023-24_matches.xlsx", sheet_name="Team IDs")
      .dropna(subset=["team_id"])
      .set_index("team_id")["raw_name (2023-24)"]
)

elo_df = (
    pd.DataFrame(ratings.items(), columns=["team_id", "elo"])
      .assign(team=lambda d: d.team_id.map(team_names))
      .sort_values("elo", ascending=False)
      .reset_index(drop=True)
)
elo_df.index += 1          # rank starting at 1
elo_df


,team_id,elo,team
1,FB,1682.130609,FENERBAHÇE OPET
2,VB,1659.411607,VAKIFBANK
3,ECZ,1657.501292,ECZACIBAŞI DYNAVİT
4,THY,1601.948270,TÜRK HAVA YOLLARI
5,KUZ,1545.601747,KUZEYBORU
6,GS,1517.189360,GALATASARAY DAIKIN
7,NIL,1469.826965,NİLÜFER BLD.
8,AYD,1445.813292,AYDIN B.ŞEHİR BLD.
9,ANK,1445.640335,ANKARAGÜCÜ
10,CUK,1437.450771,ÇUKUROVA BLD.


In [5]:
def win_prob(home, away, ratings, H=50, shared_hall=False):
    h = 0 if shared_hall else H
    return 1 / (1 + 10 ** ((ratings[away] - (ratings[home] + h)) / 400))

p = win_prob("FB", "VB", ratings)
print(f"FB (home) {p:.1%}  –  VB (away) {1-p:.1%}")


FB (home) 60.3%  –  VB (away) 39.7%


In [6]:
from scipy.optimize import brentq

def match_from_set(p):
    q = 1 - p
    return p**3 * (1 + 3*q + 6*q**2)          # P(win 3-0) + P(3-1) + P(3-2)

def set_scores(p_match):
    p = brentq(lambda x: match_from_set(x) - p_match, 1e-6, 1 - 1e-6)
    q = 1 - p
    return {"3-0": p**3, "3-1": 3*p**3*q, "3-2": 6*p**3*q**2,
            "2-3": 6*q**3*p**2, "1-3": 3*q**3*p, "0-3": q**3}

probs = set_scores(win_prob("FB", "VB", ratings))
pd.Series(probs).map("{:.1%}".format)


3-0    17.1%
3-1    22.9%
3-2    20.3%
2-3    16.3%
1-3    14.6%
0-3     8.8%
dtype: object

In [7]:
import numpy as np
rng = np.random.default_rng(42)

def simulate(home, away, ratings, n=10_000):
    p_match = win_prob(home, away, ratings)
    p = brentq(lambda x: match_from_set(x) - p_match, 1e-6, 1 - 1e-6)
    results = []
    for _ in range(n):
        h = a = 0
        while h < 3 and a < 3:
            if rng.random() < p: h += 1
            else: a += 1
        results.append(f"{h}-{a}")
    return pd.Series(results).value_counts(normalize=True)

simulate("FB", "VB", ratings)


3-1    0.2305
3-2    0.2039
3-0    0.1656
2-3    0.1626
1-3    0.1507
0-3    0.0867
Name: proportion, dtype: float64